# Fine-tuning de Embeddings no Google Colab

**Issue #2:** Fine-tuning de BGE-M3 para notícias governamentais brasileiras

**Tempo estimado:** 20-30 minutos (com GPU T4)

---

## ⚠️ Antes de Começar

1. **Ativar GPU:**
   - `Runtime` → `Change runtime type` → `Hardware accelerator` → `T4 GPU`
   
2. **Manter aba aberta:**
   - Colab desconecta se inativo por muito tempo
   - Sessão máx: 12h (mais que suficiente)

3. **Arquivos necessários:**
   - Você vai fazer upload do dataset de triplas
   - Ou conectar ao Google Drive (se dataset estiver lá)

---

## 1. Verificar GPU e Ambiente

In [ ]:
# Verificar GPU
!nvidia-smi

import torch

print(f"\n{'='*60}")
print("CONFIGURAÇÃO DO AMBIENTE")
print(f"{'='*60}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("\n✅ GPU detectada! Treino será rápido (~20-30min)")
else:
    print("\n⚠️ GPU NÃO detectada!")
    print("   Vá em: Runtime → Change runtime type → GPU")

## 2. Instalar Dependências

In [ ]:
!pip install -q sentence-transformers==2.7.0
!pip install -q pandas scikit-learn

print("✅ Dependências instaladas")

## 3. Upload do Dataset

**Opção A:** Upload manual dos arquivos CSV  
**Opção B:** Conectar ao Google Drive (se dataset estiver lá)

Escolha uma das opções abaixo:

### Opção A: Upload Manual

In [ ]:
from google.colab import files
import os

# Criar diretórios
os.makedirs('data/finetuning', exist_ok=True)

print("📤 Faça upload dos seguintes arquivos:")
print("  - train_fewshot.csv (500 triplas)")
print("  - val.csv (329 triplas)")
print("  - test.csv (370 triplas)")
print("\nOu para full dataset:")
print("  - train.csv (1668 triplas)")
print("  - val.csv")
print("  - test.csv")
print()

uploaded = files.upload()

# Mover para diretório correto
for filename in uploaded.keys():
    os.rename(filename, f'data/finetuning/{filename}')

print("\n✅ Upload concluído!")
print(f"Arquivos carregados: {list(uploaded.keys())}")

### Opção B: Google Drive (Alternativa)

In [ ]:
# Descomente se quiser usar Google Drive

# from google.colab import drive
# drive.mount('/content/drive')

# # Ajuste o path para onde seus arquivos estão no Drive
# !cp /content/drive/MyDrive/embeddings/data/finetuning/*.csv data/finetuning/

# print("✅ Arquivos copiados do Google Drive")

### Verificar Arquivos

In [ ]:
import pandas as pd

print("📂 Verificando dataset...\n")

for filename in ['train_fewshot.csv', 'train.csv', 'val.csv', 'test.csv']:
    filepath = f'data/finetuning/{filename}'
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        print(f"✅ {filename:20s}: {len(df):4d} triplas")
    else:
        print(f"⚠️  {filename:20s}: não encontrado")

print("\n📊 Preview da primeira tripla:")
if os.path.exists('data/finetuning/train_fewshot.csv'):
    df = pd.read_csv('data/finetuning/train_fewshot.csv')
elif os.path.exists('data/finetuning/train.csv'):
    df = pd.read_csv('data/finetuning/train.csv')
else:
    raise FileNotFoundError("Nenhum arquivo de treino encontrado!")

example = df.iloc[0]
print(f"\nQuery:    {example['query'][:80]}...")
print(f"Positive: {example['positive'][:80]}...")
print(f"Negative: {example['negative'][:80]}...")

## 4. Configuração do Fine-tuning

Escolha o tipo de experimento:

In [ ]:
# CONFIGURAÇÃO
# =============

# Escolha o dataset:
DATASET_TYPE = 'fewshot'  # 'fewshot' (500 triplas) ou 'full' (1668 triplas)

# Configuração de treino
CONFIG = {
    'fewshot': {
        'train_file': 'data/finetuning/train_fewshot.csv',
        'epochs': 2,
        'batch_size': 16,
        'warmup_steps': 50,
        'eval_steps': 25,
        'tempo_estimado': '20-30 minutos'
    },
    'full': {
        'train_file': 'data/finetuning/train.csv',
        'epochs': 3,
        'batch_size': 16,
        'warmup_steps': 100,
        'eval_steps': 50,
        'tempo_estimado': '1-1.5 horas'
    }
}

config = CONFIG[DATASET_TYPE]

print(f"{'='*60}")
print(f"CONFIGURAÇÃO: {DATASET_TYPE.upper()}")
print(f"{'='*60}")
print(f"Train file:       {config['train_file']}")
print(f"Épocas:           {config['epochs']}")
print(f"Batch size:       {config['batch_size']}")
print(f"Warmup steps:     {config['warmup_steps']}")
print(f"Eval steps:       {config['eval_steps']}")
print(f"Tempo estimado:   {config['tempo_estimado']}")
print()

## 5. Carregar Dados e Preparar Treino

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from torch.utils.data import DataLoader

print("📂 Carregando dados...\n")

# Load train
train_df = pd.read_csv(config['train_file'])
print(f"✅ Train: {len(train_df)} triplas")

# Load validation
val_df = pd.read_csv('data/finetuning/val.csv')
print(f"✅ Val:   {len(val_df)} triplas")

# Converter para InputExample
print("\n🔄 Preparando dados de treino...")
train_examples = []
for _, row in train_df.iterrows():
    train_examples.append(
        InputExample(texts=[row['query'], row['positive'], row['negative']])
    )

# DataLoader
train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=config['batch_size']
)

print(f"✅ {len(train_examples)} exemplos prontos")
print(f"✅ {len(train_dataloader)} batches por época")

# Preparar validation data
print("\n🔄 Preparando dados de validação...")
val_queries = {}
val_corpus = {}
val_relevant_docs = {}

for idx, row in val_df.iterrows():
    query_id = f"q{idx}"
    pos_id = f"pos_{idx}"
    neg_id = f"neg_{idx}"
    
    val_queries[query_id] = row['query']
    val_corpus[pos_id] = row['positive']
    val_corpus[neg_id] = row['negative']
    val_relevant_docs[query_id] = {pos_id}

print(f"✅ {len(val_queries)} queries de validação")
print()

## 6. Carregar Modelo Base

In [ ]:
print("📥 Carregando BGE-M3...\n")

model = SentenceTransformer('BAAI/bge-m3')

print(f"✅ Modelo carregado")
print(f"   Dimensão: {model.get_sentence_embedding_dimension()}")
print(f"   Max length: {model.max_seq_length}")
print(f"   Device: {model.device}")
print()

## 7. Configurar Loss e Evaluator

In [ ]:
# Loss function
train_loss = losses.MultipleNegativesRankingLoss(model)

print("📉 Loss: MultipleNegativesRankingLoss")
print("   (Aproxima query de positive, afasta de negative)")
print()

# Evaluator
evaluator = InformationRetrievalEvaluator(
    queries=val_queries,
    corpus=val_corpus,
    relevant_docs=val_relevant_docs,
    name='val',
    show_progress_bar=True
)

print("✅ Evaluator configurado")
print()

## 8. Fine-tuning 🚀

**Atenção:** Esta célula vai demorar ~20-30 minutos (fewshot) ou ~1-1.5h (full).

Você verá:
- Progress bar do treino
- Avaliação periódica no validation set
- Loss diminuindo (modelo aprendendo)

**Mantenha a aba aberta!**

In [ ]:
from datetime import datetime

# Output path
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = f"models/bge-m3-{DATASET_TYPE}-{timestamp}"

print(f"{'='*60}")
print("INICIANDO FINE-TUNING")
print(f"{'='*60}")
print(f"Modelo será salvo em: {output_path}")
print(f"Tempo estimado: {config['tempo_estimado']}")
print()
print("⏳ Treinando...")
print()

# TREINO
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=config['epochs'],
    warmup_steps=config['warmup_steps'],
    evaluator=evaluator,
    evaluation_steps=config['eval_steps'],
    output_path=output_path,
    save_best_model=True,
    show_progress_bar=True,
    optimizer_params={'lr': 2e-5}
)

print()
print(f"{'='*60}")
print("✅ FINE-TUNING CONCLUÍDO!")
print(f"{'='*60}")
print(f"Modelo salvo em: {output_path}")
print()

## 9. Salvar Configuração

In [ ]:
import json

training_config = {
    'base_model': 'BAAI/bge-m3',
    'dataset_type': DATASET_TYPE,
    'train_samples': len(train_examples),
    'val_samples': len(val_queries),
    'epochs': config['epochs'],
    'batch_size': config['batch_size'],
    'learning_rate': 2e-5,
    'warmup_steps': config['warmup_steps'],
    'device': str(model.device),
    'timestamp': timestamp,
    'trained_on': 'Google Colab',
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
}

with open(f'{output_path}/training_config.json', 'w') as f:
    json.dump(training_config, f, indent=2)

print("💾 Configuração salva em: training_config.json")
print()
print(json.dumps(training_config, indent=2))

## 10. Avaliação Rápida no Test Set

In [ ]:
print("🔄 Avaliando modelo fine-tuned no test set...\n")

# Carregar test set
test_df = pd.read_csv('data/finetuning/test.csv')
print(f"✅ Test set: {len(test_df)} triplas\n")

# Preparar test data
test_queries = {}
test_corpus = {}
test_relevant_docs = {}

for idx, row in test_df.iterrows():
    query_id = f"q{idx}"
    pos_id = f"pos_{idx}"
    neg_id = f"neg_{idx}"
    
    test_queries[query_id] = row['query']
    test_corpus[pos_id] = row['positive']
    test_corpus[neg_id] = row['negative']
    test_relevant_docs[query_id] = {pos_id}

# Evaluator para test
test_evaluator = InformationRetrievalEvaluator(
    queries=test_queries,
    corpus=test_corpus,
    relevant_docs=test_relevant_docs,
    name='test',
    show_progress_bar=True
)

# Avaliar
print("📊 Calculando métricas...\n")
test_score = test_evaluator(model, output_path=output_path)

print(f"\n{'='*60}")
print("RESULTADOS NO TEST SET")
print(f"{'='*60}")
print(f"Score: {test_score:.4f}")
print()
print("📁 Resultados detalhados salvos em:")
print(f"   {output_path}/test_evaluation_results.csv")
print()

## 11. Comparar com Baseline Zero-shot

In [ ]:
print("🔄 Carregando baseline zero-shot (BGE-M3 original)...\n")

baseline_model = SentenceTransformer('BAAI/bge-m3')

print("📊 Avaliando baseline no mesmo test set...\n")
baseline_score = test_evaluator(baseline_model)

print(f"\n{'='*60}")
print("COMPARAÇÃO: Fine-tuned vs Zero-shot")
print(f"{'='*60}")
print(f"Baseline (zero-shot): {baseline_score:.4f}")
print(f"Fine-tuned:           {test_score:.4f}")
print()
delta = test_score - baseline_score
delta_pct = (delta / baseline_score) * 100 if baseline_score > 0 else 0

indicator = "✅" if delta > 0.01 else ("❌" if delta < -0.01 else "➖")

print(f"Δ Absoluto: {delta:+.4f} {indicator}")
print(f"Δ %:        {delta_pct:+.2f}%")
print()

if delta > 0.02:
    print("🎉 SUCESSO! Fine-tuning melhorou significativamente (+2% ou mais)")
    print("   Vale escalar para full dataset ou mais épocas.")
elif delta > 0.005:
    print("✅ Ganho modesto. Fine-tuning ajudou um pouco.")
    print("   Considerar se ROI compensa.")
else:
    print("⚠️  Ganho mínimo ou inexistente.")
    print("   Fine-tuning pode não valer a pena para este domínio.")
    print("   Ou dataset é muito pequeno.")

## 12. Download do Modelo Treinado

In [ ]:
import shutil

# Criar arquivo ZIP do modelo
print("📦 Criando arquivo ZIP do modelo...\n")

zip_filename = f"bge-m3-{DATASET_TYPE}-{timestamp}"
shutil.make_archive(zip_filename, 'zip', output_path)

zip_path = f"{zip_filename}.zip"
zip_size = os.path.getsize(zip_path) / (1024 * 1024)  # MB

print(f"✅ ZIP criado: {zip_path}")
print(f"   Tamanho: {zip_size:.1f} MB")
print()

# Download
print("📥 Iniciando download...")
print("   (Pode demorar alguns minutos, aguarde)\n")

files.download(zip_path)

print("\n✅ Download iniciado!")
print("   Verifique a pasta de downloads do seu navegador.")
print()
print("📁 Para usar o modelo localmente:")
print("   1. Extrair o ZIP")
print("   2. Copiar para: models/bge-m3-finetuned/")
print("   3. Carregar com: SentenceTransformer('models/bge-m3-finetuned')")

## 13. Salvar Resultados (JSON)

In [ ]:
results = {
    'experiment': {
        'dataset_type': DATASET_TYPE,
        'timestamp': timestamp,
        'trained_on': 'Google Colab',
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
    },
    'training': training_config,
    'evaluation': {
        'test_samples': len(test_queries),
        'baseline_score': float(baseline_score),
        'finetuned_score': float(test_score),
        'delta_absolute': float(delta),
        'delta_percent': float(delta_pct),
        'improvement': 'significant' if delta > 0.02 else ('modest' if delta > 0.005 else 'minimal')
    },
    'files': {
        'model_path': output_path,
        'zip_file': zip_path,
        'config': f'{output_path}/training_config.json',
        'results': f'{output_path}/test_evaluation_results.csv'
    }
}

results_file = f'results_{DATASET_TYPE}_{timestamp}.json'
with open(results_file, 'w') as f:
    json.dump(results, f, indent=2)

print(f"💾 Resultados salvos em: {results_file}\n")
print(json.dumps(results, indent=2))

# Download results
print(f"\n📥 Baixando {results_file}...")
files.download(results_file)

## 14. Resumo Final

In [ ]:
print(f"{'='*60}")
print("RESUMO DO EXPERIMENTO")
print(f"{'='*60}")
print()
print(f"Dataset:        {DATASET_TYPE.upper()} ({len(train_examples)} triplas)")
print(f"Épocas:         {config['epochs']}")
print(f"Device:         {training_config['gpu']}")
print()
print(f"{'='*60}")
print("RESULTADOS")
print(f"{'='*60}")
print(f"Baseline:       {baseline_score:.4f}")
print(f"Fine-tuned:     {test_score:.4f}")
print(f"Improvement:    {delta:+.4f} ({delta_pct:+.2f}%) {indicator}")
print()
print(f"{'='*60}")
print("ARQUIVOS GERADOS")
print(f"{'='*60}")
print(f"Modelo ZIP:     {zip_path} ({zip_size:.1f} MB)")
print(f"Resultados:     {results_file}")
print(f"Config:         {output_path}/training_config.json")
print()
print(f"{'='*60}")
print("PRÓXIMOS PASSOS")
print(f"{'='*60}")
print()

if delta > 0.02:
    print("✅ Fine-tuning funcionou!")
    print("\nRecomendações:")
    print("  1. Se rodou fewshot, escalar para full dataset")
    print("  2. Testar com mais épocas (4-5)")
    print("  3. Expandir dataset (10k-20k triplas)")
    print("  4. Documentar ganhos na Issue #2")
elif delta > 0.005:
    print("➖ Ganho modesto")
    print("\nRecomendações:")
    print("  1. Análise qualitativa: onde melhorou?")
    print("  2. Testar em modelos mais fracos (BERTimbau)")
    print("  3. Considerar ROI: vale expandir dataset?")
    print("  4. Explorar Issue #3 (Hybrid Search)")
else:
    print("⚠️  Fine-tuning não trouxe ganhos")
    print("\nRecomendações:")
    print("  1. Baseline (zero-shot) continua sendo melhor")
    print("  2. Focar em Issue #3 (Hybrid Search)")
    print("  3. Ou Issue #5 (RAG)")
    print("  4. Documentar que fine-tuning não compensa")

print()
print(f"{'='*60}")
print("✅ EXPERIMENTO CONCLUÍDO!")
print(f"{'='*60}")

---

## 📚 Documentação

- **Issue #2:** Fine-tuning vs Zero-shot
- **FINETUNING_GUIDE.md:** Guia completo de fine-tuning
- **Repo:** https://github.com/destaquesgovbr/data-science

---

**Desenvolvido por:** Luis Felipe de Moraes  
**Data:** 2026-04-14